### Scree plots and PC plots for all Dementia Subtypes

In [ ]:
#!/usr/bin/env python3

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from itertools import combinations

# directories 
working = "/scratch/c.c2029098/dementia_ml_project"

pairs = ["BDR_AD_control", "VaD_AD", "LBD_AD"]  

N_PCS         = 7      
JITTER        = 1e-3
MARKER_SIZE   = 28
ALPHA         = 0.7

# Outliers for plotting 
K_PCS_FOR_Z   = 10
OUTLIER_Z     = 6.0
DROP_OUTLIERS = True

# Paths
pca_root      = f"{working}/results/PCA"
prs_profile   = f"{working}/results/PRS/BDR_AD_PRS.profile"  # optional; set to None to skip
ml_data_root  = f"{working}/data/processed/ml_data"


def load_eigvec(prefix):
    """Read <prefix>.eigenvec with first two cols as FID/IID and the rest PCs."""
    df = pd.read_csv(f"{prefix}.eigenvec", sep=r"\s+", engine="python")
    cols = list(df.columns)
    # Force simple naming: FID, IID, PC1..PCm
    df.columns = ["FID", "IID"] + [f"PC{i}" for i in range(1, len(cols) - 1)]
    return df, [c for c in df.columns if c.startswith("PC")]

def load_eigval(prefix):
    """Read <prefix>.eigenval and compute prop variance."""
    ev = pd.read_csv(f"{prefix}.eigenval", delim_whitespace=True, header=None, names=["eigenvalue"])
    ev["prop_var"] = ev["eigenvalue"] / ev["eigenvalue"].sum()
    ev["PC"] = np.arange(1, len(ev) + 1)
    return ev

def load_fam(path):
    """Read <pair>.fam -> FID, IID, PHENOTYPE (1=control, 2=case; -9/0 treated as NA)."""
    fam = pd.read_csv(path, delim_whitespace=True, header=None,
                      names=["FID","IID","PID","MID","SEX","PHENOTYPE"], dtype=str)
    fam = fam[["FID","IID","PHENOTYPE"]].copy()
    fam["PHENOTYPE"] = pd.to_numeric(fam["PHENOTYPE"], errors="coerce")
    fam.loc[fam["PHENOTYPE"].isin([-9, 0]), "PHENOTYPE"] = np.nan
    return fam

def load_prs(profile_path):
    """Read PRS profile and standardize column name to PRS (optional)."""
    prof = pd.read_csv(profile_path, delim_whitespace=True)
    cand = [c for c in prof.columns if "SCORE" in c.upper()] or (["SCORESUM"] if "SCORESUM" in prof.columns else [])
    prs_col = cand[0] if cand else None
    if prs_col:
        prof = prof.rename(columns={prs_col: "PRS"})
        return prof[["FID","IID","PRS"]]
    return None  # nothing PRS-like found

def labels_from_pheno(df):
    return df["PHENOTYPE"].map({1.0: "Control", 2.0: "Case"}) if "PHENOTYPE" in df.columns else None

def zflag(df, pc_cols, k=10, zthr=6.0):
    """Flag samples having |z|>zthr on any of first k PCs (standard z-score)."""
    pcs = pc_cols[:min(k, len(pc_cols))]
    X = df[pcs].to_numpy(float)
    Z = (X - np.nanmean(X, 0)) / np.nanstd(X, 0, ddof=0)
    return (np.abs(Z) > zthr).any(axis=1)

def plot_pair(df, labels, a, b, title, out_dir):
    x = df[a].to_numpy()
    y = df[b].to_numpy()
    xj = x + np.random.normal(0, JITTER, size=len(x))
    yj = y + np.random.normal(0, JITTER, size=len(y))

    plt.figure(figsize=(7, 6))
    if labels is None or labels.isna().all():
        plt.scatter(xj, yj, s=MARKER_SIZE, alpha=ALPHA, edgecolors="none")
    else:
        for lab in sorted(labels.dropna().unique()):
            mask = (labels == lab).to_numpy()
            plt.scatter(xj[mask], yj[mask], s=MARKER_SIZE, alpha=ALPHA, edgecolors="none", label=str(lab))
        if labels.isna().any():
            m = labels.isna().to_numpy()
            plt.scatter(xj[m], yj[m], s=MARKER_SIZE, alpha=ALPHA, edgecolors="none", label="Missing")
        plt.legend(title="Phenotype", frameon=False)

    plt.xlabel(a); plt.ylabel(b)
    plt.title(f"{title}: {a} vs {b} (jittered, outliers {'dropped' if DROP_OUTLIERS else 'kept'})")
    plt.tight_layout()
    os.makedirs(out_dir, exist_ok=True)
    out = os.path.join(out_dir, f"{a}_{b}.png")
    plt.savefig(out, dpi=200)
    plt.close()
    return out

def process_pair(pair):
    print(f"\n {pair} ")

    prefix   = f"{pca_root}/{pair}"
    plotsdir = f"{pca_root}/{pair}/plots"
    os.makedirs(plotsdir, exist_ok=True)
    os.makedirs(ml_data_root, exist_ok=True)

    # Load PCA
    ev, pc_cols = load_eigvec(prefix)
    eigval = load_eigval(prefix)

    #  Merge phenotype
    fam = load_fam(f"{working}/data/processed/genotypes/{pair}.fam")
    df = ev.merge(fam, on=["FID","IID"], how="left")

    # Merge PRS with PC's
    if prs_profile and os.path.exists(prs_profile):
        prs = load_prs(prs_profile)
        if prs is not None:
            df = df.merge(prs, on=["FID","IID"], how="left")

    # export PC's and PRS merged files
    cols = ["FID","IID"] + (["PHENOTYPE"] if "PHENOTYPE" in df.columns else []) + pc_cols + (["PRS"] if "PRS" in df.columns else [])
    slim = df[cols].copy()
    slim.to_csv(f"{ml_data_root}/{pair}_PCs_PRS.tsv", sep="\t", index=False)
    slim.to_csv(f"{ml_data_root}/{pair}_PCs_PRS.raw", sep=" ", index=False)  # space-delimited, PLINK-style

    # If there is a genotype .raw already, make a simple merged ML file
    geno_raw = f"{ml_data_root}/{pair}.raw"
    if os.path.exists(geno_raw):
        geno = pd.read_csv(geno_raw, delim_whitespace=True)
        merged = geno.merge(slim, on=["FID","IID"], how="left")
        merged.to_csv(f"{ml_data_root}/{pair}_merged_ml.raw", sep=" ", index=False)

    # Outliers (for plot only)
    out_mask = zflag(df, pc_cols, k=K_PCS_FOR_Z, zthr=OUTLIER_Z)
    plot_df = df.loc[~out_mask].copy() if DROP_OUTLIERS else df.copy()
    labels  = labels_from_pheno(plot_df)

    # Pairwise PC plots
    use_pcs = pc_cols[:min(N_PCS, len(pc_cols))]
    made = [plot_pair(plot_df, labels, a, b, pair, plotsdir) for a, b in combinations(use_pcs, 2)]
    print(f"Saved {len(made)} PC pair plots to {plotsdir}")

   # Scree plot
    plt.figure(figsize=(7, 5))
    plt.plot(eigval["PC"], eigval["prop_var"], marker="o")
    plt.xlabel("Principal Component")
    plt.ylabel("Proportion of variance explained")
    plt.title(f"{pair}: Scree")
    plt.xticks(eigval["PC"])
    plt.tight_layout()
    plt.savefig(f"{plotsdir}/scree.png", dpi=200)
    plt.close()
    print(f"Scree: {plotsdir}/scree.png")

def main():
    for p in pairs:
        process_pair(p)

main()


 BDR_AD_control 


/tmp/ipykernel_3170/2690150704.py:42: FutureWarning: The 'delim_whitespace' keyword in pd.read_csv is deprecated and will be removed in a future version. Use ``sep='\s+'`` instead
  ev = pd.read_csv(f"{prefix}.eigenval", delim_whitespace=True, header=None, names=["eigenvalue"])
/tmp/ipykernel_3170/2690150704.py:49: FutureWarning: The 'delim_whitespace' keyword in pd.read_csv is deprecated and will be removed in a future version. Use ``sep='\s+'`` instead
  fam = pd.read_csv(path, delim_whitespace=True, header=None,
/tmp/ipykernel_3170/2690150704.py:58: FutureWarning: The 'delim_whitespace' keyword in pd.read_csv is deprecated and will be removed in a future version. Use ``sep='\s+'`` instead
  prof = pd.read_csv(profile_path, delim_whitespace=True)
/tmp/ipykernel_3170/2690150704.py:134: FutureWarning: The 'delim_whitespace' keyword in pd.read_csv is deprecated and will be removed in a future version. Use ``sep='\s+'`` instead
  geno = pd.read_csv(geno_raw, delim_whitespace=True)


Saved 21 PC pair plots to /scratch/c.c2029098/dementia_ml_project/results/PCA/BDR_AD_control/plots
Scree: /scratch/c.c2029098/dementia_ml_project/results/PCA/BDR_AD_control/plots/scree.png

 VaD_AD 


/tmp/ipykernel_3170/2690150704.py:42: FutureWarning: The 'delim_whitespace' keyword in pd.read_csv is deprecated and will be removed in a future version. Use ``sep='\s+'`` instead
  ev = pd.read_csv(f"{prefix}.eigenval", delim_whitespace=True, header=None, names=["eigenvalue"])
/tmp/ipykernel_3170/2690150704.py:49: FutureWarning: The 'delim_whitespace' keyword in pd.read_csv is deprecated and will be removed in a future version. Use ``sep='\s+'`` instead
  fam = pd.read_csv(path, delim_whitespace=True, header=None,
/tmp/ipykernel_3170/2690150704.py:58: FutureWarning: The 'delim_whitespace' keyword in pd.read_csv is deprecated and will be removed in a future version. Use ``sep='\s+'`` instead
  prof = pd.read_csv(profile_path, delim_whitespace=True)
/tmp/ipykernel_3170/2690150704.py:134: FutureWarning: The 'delim_whitespace' keyword in pd.read_csv is deprecated and will be removed in a future version. Use ``sep='\s+'`` instead
  geno = pd.read_csv(geno_raw, delim_whitespace=True)


Saved 21 PC pair plots to /scratch/c.c2029098/dementia_ml_project/results/PCA/VaD_AD/plots
Scree: /scratch/c.c2029098/dementia_ml_project/results/PCA/VaD_AD/plots/scree.png

 LBD_AD 


/tmp/ipykernel_3170/2690150704.py:42: FutureWarning: The 'delim_whitespace' keyword in pd.read_csv is deprecated and will be removed in a future version. Use ``sep='\s+'`` instead
  ev = pd.read_csv(f"{prefix}.eigenval", delim_whitespace=True, header=None, names=["eigenvalue"])
/tmp/ipykernel_3170/2690150704.py:49: FutureWarning: The 'delim_whitespace' keyword in pd.read_csv is deprecated and will be removed in a future version. Use ``sep='\s+'`` instead
  fam = pd.read_csv(path, delim_whitespace=True, header=None,
/tmp/ipykernel_3170/2690150704.py:58: FutureWarning: The 'delim_whitespace' keyword in pd.read_csv is deprecated and will be removed in a future version. Use ``sep='\s+'`` instead
  prof = pd.read_csv(profile_path, delim_whitespace=True)
/tmp/ipykernel_3170/2690150704.py:134: FutureWarning: The 'delim_whitespace' keyword in pd.read_csv is deprecated and will be removed in a future version. Use ``sep='\s+'`` instead
  geno = pd.read_csv(geno_raw, delim_whitespace=True)


Saved 21 PC pair plots to /scratch/c.c2029098/dementia_ml_project/results/PCA/LBD_AD/plots
Scree: /scratch/c.c2029098/dementia_ml_project/results/PCA/LBD_AD/plots/scree.png
